In [23]:
import time
import json
import numpy as np
import pandas as pd
import faiss
from pypdf import PdfReader
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import pipeline
from rank_bm25 import BM25Okapi
from google.colab import files

In [24]:
def ingest_pdf(path):
    """Load a PDF and return raw text."""
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

def ingest_txt(path):
    """Load a plain text file and return raw text."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def ingest_hf_dataset(dataset_name, split="train", text_column="text", max_rows=200):
    """Load a Hugging Face dataset and concatenate a text column into raw text."""
    ds = load_dataset(dataset_name, split=split)
    rows = ds.select(range(min(max_rows, len(ds))))
    combined_text = "\n".join(row[text_column] for row in rows if row.get(text_column))
    return combined_text

def ingest_document(source_type, source_path_or_name, **kwargs):
    """
    Unified ingestion router.
    source_type: 'pdf', 'txt', or 'hf'
    """
    if source_type == "pdf":
        return ingest_pdf(source_path_or_name)
    elif source_type == "txt":
        return ingest_txt(source_path_or_name)
    elif source_type == "hf":
        return ingest_hf_dataset(source_path_or_name, **kwargs)
    else:
        raise ValueError("source_type must be 'pdf', 'txt', or 'hf'")

In [25]:
uploaded = files.upload()  # upload a .pdf or .txt file
file_path = list(uploaded.keys())[0]

source_type = "pdf" if file_path.lower().endswith(".pdf") else "txt"
raw_text = ingest_document(source_type, file_path)

print(f"Ingested {len(raw_text)} characters from {file_path}")
print(raw_text[:400])

Saving AI ML PREPARATION.pdf to AI ML PREPARATION.pdf
Ingested 81285 characters from AI ML PREPARATION.pdf
T H E  C O M P L E T E
AI / ML Interview
Preparation Guide
A field-tested playbook for answering with structure, intuition & confidence
Classical ML Deep Learning NLP & Transformers LLMs & GenAI 
46 Concepts  ·  4 Domains  ·  Interview-Ready
Every concept follows one flow: how to start answering → the math intuition → pros & 
cons → the follow-ups interviewers actually ask.
Ravindra Sindhiya
Sr. A


In [27]:
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    """Split raw text into overlapping chunks for better retrieval granularity."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    return splitter.split_text(text)

chunks = chunk_text(raw_text, chunk_size=500, chunk_overlap=50)
print(f"Created {len(chunks)} chunks")
print(chunks[0])

Created 180 chunks
T H E  C O M P L E T E
AI / ML Interview
Preparation Guide
A field-tested playbook for answering with structure, intuition & confidence
Classical ML Deep Learning NLP & Transformers LLMs & GenAI 
46 Concepts  ·  4 Domains  ·  Interview-Ready
Every concept follows one flow: how to start answering → the math intuition → pros & 
cons → the follow-ups interviewers actually ask.
Ravindra Sindhiya
Sr. AI / ML Engineer
in linkedin.com/in/ravindra-sindhiya-b08609153


In [28]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunks(chunks, model=embed_model):
    """Convert a list of text chunks into vector embeddings."""
    embeddings = model.encode(chunks, show_progress_bar=True)
    return np.array(embeddings).astype("float32")

chunk_embeddings = embed_chunks(chunks)
print(f"Embedding matrix shape: {chunk_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix shape: (180, 384)


In [29]:
def build_vector_store(embeddings):
    """Initialize a FAISS index and load embeddings for fast similarity search."""
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)  # exact L2 similarity search
    index.add(embeddings)
    return index

vector_store = build_vector_store(chunk_embeddings)
print(f"Vector store initialized with {vector_store.ntotal} vectors")

Vector store initialized with 180 vectors


In [30]:
def encode_query(query, model=embed_model):
    """Convert an incoming user question into its vector representation."""
    return model.encode([query]).astype("float32")

In [31]:
def retrieve_chunks(query, index, chunks, top_k=3):
    """Retrieve the most contextually relevant chunks for a given query."""
    query_vector = encode_query(query)
    distances, indices = index.search(query_vector, top_k)
    results = [(chunks[i], float(distances[0][pos])) for pos, i in enumerate(indices[0])]
    return results

# quick test
results = retrieve_chunks("What is the main idea of the document?", vector_store, chunks)
for i, (text, score) in enumerate(results):
    print(f"--- Result {i+1} (distance={score:.3f}) ---\n{text}\n")

--- Result 1 (distance=1.390) ---
in linkedin.com/in/ravindra-sindhiya-b08609153 
▲ Click the link above to connect with me on LinkedIn
 
How to use this guide
Every concept follows the same four-part structure so you can answer any interview 
question with a clear arc:
• How to start answering — a crisp 1–2 sentence opener that frames the concept 
before you dive into detail. Interviewers judge structure as much as content. 
• Math intuition — the core equation and, more importantly, what it means. State

--- Result 2 (distance=1.467) ---
assumption, it's fast, scales beautifully, and is a strong baseline for text classification."
MATH INTUITION
Bayes: P(y | x) ∝ P(y) · Πj P(xj | y)
Intuition: we want the most probable class given the features. By Bayes' rule that's 
proportional to the prior times the likelihood. The naive independence assumption lets us 
multiply per-feature likelihoods instead of modelling their joint distribution — turning a hard

--- Result 3 (distance=1.469) ---

In [46]:
generator = pipeline("text-generation", model="google/flan-t5-base")

def build_prompt(query, retrieved_chunks):
    """Combine retrieved context and the original query into one grounded prompt."""
    context = "\n\n".join(text for text, _ in retrieved_chunks)
    prompt = f"""Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
{context}

Question: {query}

Answer:"""
    return prompt

def generate_answer(query, index, chunks, top_k=3):
    retrieved = retrieve_chunks(query, index, chunks, top_k)
    prompt = build_prompt(query, retrieved)
    result = generator(prompt, max_length=200, do_sample=False)
    return result[0]["generated_text"], retrieved

answer, sources = generate_answer("What is the main idea of the document?", vector_store, chunks)
print("Answer:", answer)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

Answer: Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
in linkedin.com/in/ravindra-sindhiya-b08609153 
▲ Click the link above to connect with me on LinkedIn
 
How to use this guide
Every concept follows the same four-part structure so you can answer any interview 
question with a clear arc:
• How to start answering — a crisp 1–2 sentence opener that frames the concept 
before you dive into detail. Interviewers judge structure as much as content. 
• Math intuition — the core equation and, more importantly, what it means. State

assumption, it's fast, scales beautifully, and is a strong baseline for text classification."
MATH INTUITION
Bayes: P(y | x) ∝ P(y) · Πj P(xj | y)
Intuition: we want the most probable class given the features. By Bayes' rule that's 
proportional to the prior times the likelihood. The naive independence assumption lets us 
multiply per-feature likelihoods instead of modellin

In [47]:
def run_chunk_experiment(chunk_size, chunk_overlap, query):
    exp_chunks = chunk_text(raw_text, chunk_size, chunk_overlap)
    exp_embeddings = embed_chunks(exp_chunks)
    exp_index = build_vector_store(exp_embeddings)
    answer, _ = generate_answer(query, exp_index, exp_chunks)
    return answer

print("Chunk boundary experiments:\n")
for size, overlap in [(300, 30), (500, 50), (800, 100)]:
    print(f"chunk_size={size}, overlap={overlap}")
    print(run_chunk_experiment(size, overlap, "What is ?"))
    print("-"*60)

Chunk boundary experiments:

chunk_size=300, overlap=30


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
on context windows. Efficient variants (FlashAttention for IO-efficiency, sparse/linear 
attention, sliding-window) reduce the memory or compute to make long contexts feasible.
35. Multi-Head Attention & Positional Encoding Transformers
HOW TO START ANSWERING

tokens left-to-right. It has no causal structure for autoregressive generation, so it's used 
for understanding/embedding tasks, while GPT-style causal models handle generation.
Q: What is the [CLS] token in BERT?

its assigned points. It's the go-to for fast, simple clustering when you roughly know k."
MATH INTUITION
Minimise within-cluster sum of squares (inertia): J = Σk Σx∈Ck ||x − μk||2
Intuition: it's an alternating optimisation (a form of EM). Step 1 (assign): each point joins

Question: What is ?

Answer:
------------------------------------------------------------
chunk_size=500, overl

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
MATH INTUITION
Pretraining: next-token prediction over trillions of tokens — maximise Σ log P(xt | x<t).
RLHF: train a reward model on human preference pairs, then optimise the LLM (via PPO) to 
maximise that reward with a KL penalty to stay near the original model.
Intuition: pretraining is where almost all knowledge is learned — the model compresses 
the internet by predicting the next token. But a raw pretrained model just continues text; SFT

Intuition: instead of relying on parametric memory (knowledge frozen in weights), we give 
the model a "open-book exam" — fetch the relevant passages and let it read them. Retrieval 
uses embeddings so semantically similar text is found even without keyword overlap. This 
separates knowledge (easily updated in the database) from reasoning (the LLM), and lets the 
model cite sources.
PROS
• Reduces hallucinat

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
prompting, graduate to fine-tuning if needed.
41. Retrieval-Augmented Generation (RAG) LLM Systems
HOW TO START ANSWERING
"RAG grounds an LLM in external, up-to-date knowledge by retrieving relevant documents 
and injecting them into the prompt before generation. It's the standard fix for hallucination 
and stale knowledge, and it lets the model use private/domain data it was never trained 
on."
MATH INTUITION
Pipeline: embed the query → vector-search a document store (cosine similarity) → retrieve top-
k chunks → prepend them to the prompt → the LLM answers grounded in that context.
Intuition: instead of relying on parametric memory (knowledge frozen in weights), we give 
the model a "open-book exam" — fetch the relevant passages and let it read them. Retrieval

object — a word, user, product, or category — where similar items end up close together 

In [48]:
tokenized_chunks = [c.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def hybrid_retrieve(query, index, chunks, top_k=3, alpha=0.5):
    """
    alpha=1.0 -> pure vector search
    alpha=0.0 -> pure keyword (BM25) search
    """
    query_vector = encode_query(query)
    distances, indices = index.search(query_vector, len(chunks))
    vector_scores = np.zeros(len(chunks))
    for pos, i in enumerate(indices[0]):
        vector_scores[i] = 1 / (1 + distances[0][pos])

    bm25_scores = np.array(bm25.get_scores(query.lower().split()))
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()
    if vector_scores.max() > 0:
        vector_scores = vector_scores / vector_scores.max()

    combined = alpha * vector_scores + (1 - alpha) * bm25_scores
    top_indices = np.argsort(combined)[::-1][:top_k]
    return [(chunks[i], combined[i]) for i in top_indices]

hybrid_results = hybrid_retrieve("What is the main idea of the document?", vector_store, chunks, alpha=0.5)
print("Hybrid search results:\n")
for text, score in hybrid_results:
    print(f"score={score:.3f} | {text[:150]}...")

Hybrid search results:

score=0.926 | makes the calls reliable and parseable.
46. Generative Models: GANs, VAEs & Diffusion GenAI
HOW TO START ANSWERING
"These are the main families of gen...
score=0.890 | dimensional space, using a kernel function, without ever materialising those coordinates. 
So we get non-linear boundaries at the cost of a cheap kern...
score=0.881 | • Scale with data and compute.
CONS
• Data- and compute-hungry.
• Black-box; hard to interpret.
• Many hyperparameters; prone to overfitting.
Q: Why d...


In [49]:
def run_pipeline_demo(queries, index, chunks, top_k=3):
    """Runs the full pipeline for test queries and prints grounded answers."""
    print("="*70)
    print("END-TO-END RAG PIPELINE — DEMO RUN")
    print("="*70)
    for q in queries:
        answer, retrieved = generate_answer(q, index, chunks, top_k)
        print(f"\nQ: {q}")
        print(f"A: {answer}")
        print("Grounded on chunks:")
        for i, (text, score) in enumerate(retrieved):
            print(f"   [{i+1}] (dist={score:.3f}) {text[:120]}...")
        print("-"*70)

demo_queries = [
    "What is the main idea of the document?",
    "What are the key components described?",
    "What problem does this document solve?"
]

run_pipeline_demo(demo_queries, vector_store, chunks)

END-TO-END RAG PIPELINE — DEMO RUN


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is the main idea of the document?
A: Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
in linkedin.com/in/ravindra-sindhiya-b08609153 
▲ Click the link above to connect with me on LinkedIn
 
How to use this guide
Every concept follows the same four-part structure so you can answer any interview 
question with a clear arc:
• How to start answering — a crisp 1–2 sentence opener that frames the concept 
before you dive into detail. Interviewers judge structure as much as content. 
• Math intuition — the core equation and, more importantly, what it means. State

assumption, it's fast, scales beautifully, and is a strong baseline for text classification."
MATH INTUITION
Bayes: P(y | x) ∝ P(y) · Πj P(xj | y)
Intuition: we want the most probable class given the features. By Bayes' rule that's 
proportional to the prior times the likelihood. The naive independence assumption lets us 
multiply per-f

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What are the key components described?
A: Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
in linkedin.com/in/ravindra-sindhiya-b08609153 
▲ Click the link above to connect with me on LinkedIn
 
How to use this guide
Every concept follows the same four-part structure so you can answer any interview 
question with a clear arc:
• How to start answering — a crisp 1–2 sentence opener that frames the concept 
before you dive into detail. Interviewers judge structure as much as content. 
• Math intuition — the core equation and, more importantly, what it means. State

• Removes correlated features; aids visualisation (2D/3D).
• De-noises data.
CONS
• Components are linear combinations — hard to interpret.
• Only captures linear structure.
• Sensitive to scaling; variance ≠ predictive importance.
Q: Should you scale before PCA?
A: Yes — PCA chases variance, so a feature with a large numeric range woul

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What problem does this document solve?
A: Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
Mitigations: grounding via RAG, lowering temperature, asking for citations, prompting it to 
say "I don't know," and verification/self-consistency. Evaluation can't use a single accuracy 
number — we use benchmarks (MMLU), human/LLM-as-judge ratings, and task-specific 
metrics.
MITIGATIONS
• RAG grounding + citations.
• Lower temperature; constrained decoding.
• Self-consistency / verification passes.
EVAL CHALLENGES
• No single ground-truth for open generation.

which tool to call and interpret the result. By interleaving reasoning ("Thought") with tool calls 
("Action") and feeding outputs back in ("Observation"), it decomposes complex tasks and 
grounds itself in real results — turning a static text predictor into something that acts. 
Memory and planning modules extend this to long-horizon tasks.
PROS

In [50]:
def validate_retrieval(test_queries, expected_keywords, index, chunks, top_k=3):
    """
    Checks whether expected keywords appear in retrieved chunks
    (proxy for retrieval accuracy) and logs timing for auditability.
    """
    logs = []
    for query, keywords in zip(test_queries, expected_keywords):
        start = time.time()
        retrieved = retrieve_chunks(query, index, chunks, top_k)
        elapsed = time.time() - start

        retrieved_text = " ".join(text.lower() for text, _ in retrieved)
        hits = [kw for kw in keywords if kw.lower() in retrieved_text]
        accuracy = len(hits) / len(keywords) if keywords else None

        logs.append({
            "query": query,
            "expected_keywords": keywords,
            "matched_keywords": hits,
            "keyword_hit_rate": round(accuracy, 2) if accuracy is not None else "N/A",
            "top_chunk_preview": retrieved[0][0][:100] if retrieved else "",
            "retrieval_time_sec": round(elapsed, 4)
        })

    return pd.DataFrame(logs)

# NOTE: adjust these keywords to match your actual document's content
test_queries = [
    "What is the main idea of the document?",
    "What are the stages of the system architecture?",
    "What components are used in this project?"
]
expected_keywords = [
    ["main idea", "document"],
    ["ingestion", "chunking", "embedding", "retrieval"],
    ["embedding model", "vector store", "language model"]
]

validation_log = validate_retrieval(test_queries, expected_keywords, vector_store, chunks)
print(validation_log.to_string(index=False))

validation_log.to_csv("validation_log.csv", index=False)
print("\nSaved: validation_log.csv")

                                          query                               expected_keywords matched_keywords  keyword_hit_rate                                                                                      top_chunk_preview  retrieval_time_sec
         What is the main idea of the document?                           [main idea, document]               []               0.0  in linkedin.com/in/ravindra-sindhiya-b08609153 \n▲ Click the link above to connect with me on LinkedI              0.0307
What are the stages of the system architecture?     [ingestion, chunking, embedding, retrieval]               []               0.0 36. The Transformer Architecture Transformers\nHOW TO START ANSWERING\n"The Transformer, from 'Attenti              0.0253
      What components are used in this project? [embedding model, vector store, language model]               []               0.0  which tool to call and interpret the result. By interleaving reasoning ("Thought") with tool calls \n     

In [51]:
def generate_system_report(chunks, chunk_embeddings, index):
    report = {
        "Chunking Profile": {
            "chunk_size (chars)": 500,
            "chunk_overlap (chars)": 50,
            "total_chunks": len(chunks),
            "avg_chunk_length": round(np.mean([len(c) for c in chunks]), 1),
            "splitter": "RecursiveCharacterTextSplitter (LangChain)"
        },
        "Embedding Model": {
            "model_name": "all-MiniLM-L6-v2 (sentence-transformers)",
            "embedding_dimension": chunk_embeddings.shape[1],
            "total_vectors": chunk_embeddings.shape[0]
        },
        "Vector Store": {
            "tool": "FAISS",
            "index_type": "IndexFlatL2 (exact L2 search)",
            "vectors_stored": index.ntotal
        },
        "Language Model": {
            "model_name": "google/flan-t5-base (Hugging Face)",
            "task": "text2text-generation",
            "max_output_tokens": 200,
            "decoding": "greedy (do_sample=False)"
        }
    }
    return report

metrics_report = generate_system_report(chunks, chunk_embeddings, vector_store)

print("="*70)
print("SYSTEM METRICS REPORT")
print("="*70)
for section, details in metrics_report.items():
    print(f"\n{section}:")
    for k, v in details.items():
        print(f"   {k}: {v}")

with open("system_metrics_report.json", "w") as f:
    json.dump(metrics_report, f, indent=2)
print("\nSaved: system_metrics_report.json")

SYSTEM METRICS REPORT

Chunking Profile:
   chunk_size (chars): 500
   chunk_overlap (chars): 50
   total_chunks: 180
   avg_chunk_length: 461.2
   splitter: RecursiveCharacterTextSplitter (LangChain)

Embedding Model:
   model_name: all-MiniLM-L6-v2 (sentence-transformers)
   embedding_dimension: 384
   total_vectors: 180

Vector Store:
   tool: FAISS
   index_type: IndexFlatL2 (exact L2 search)
   vectors_stored: 180

Language Model:
   model_name: google/flan-t5-base (Hugging Face)
   task: text2text-generation
   max_output_tokens: 200
   decoding: greedy (do_sample=False)

Saved: system_metrics_report.json


In [52]:
while True:
    user_query = input("\nAsk a question about your document (or type 'exit'): ")
    if user_query.lower() == "exit":
        break
    answer, _ = generate_answer(user_query, vector_store, chunks)
    print(f"\nAnswer: {answer}")


Ask a question about your document (or type 'exit'): What is svm


[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
9. Support Vector Machines (SVM) Classification
HOW TO START ANSWERING
"An SVM finds the decision boundary that maximises the margin — the distance between 
the boundary and the nearest points of each class (the support vectors). Maximising this 
margin gives strong generalisation, and the kernel trick lets it handle non-linear 
boundaries."
MATH INTUITION
Maximise the margin 2 / ||w||, i.e. minimise ½||w||2 subject to yi(wTxi + b) ≥ 1

dimensional space, using a kernel function, without ever materialising those coordinates. 
So we get non-linear boundaries at the cost of a cheap kernel evaluation. RBF is the 
common default.
Q: What do C and gamma control in an RBF SVM?
A: C is the regularisation / margin-softness trade-off (high C = fit training data tightly). 
Gamma is the reach of a single training point (high gamma = tight, wiggly bound

[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer: Answer the question using only the context below.
If the answer isn't in the context, say "I don't know based on the document."

Context:
connecting every pixel to every neuron, they slide small learnable filters across the input 
to detect local patterns, sharing weights to exploit the fact that a feature is useful 
anywhere in the image."
MATH INTUITION
Convolution: a filter (kernel) slides over the input computing dot products — (I * K)(i,j) = ΣΣ 
I(i+m, j+n) K(m,n)
Intuition: three ideas make CNNs efficient. Local receptive fields — nearby pixels are

A: A dense layer on a 224×224 image would need billions of weights. CNNs share a small 
filter across all positions, cutting parameters massively and building in the prior that 
visual features are local and translation-invariant — which also reduces overfitting.
28. RNNs, LSTMs & GRUs Sequence Models
HOW TO START ANSWERING
"RNNs process sequences by maintaining a hidden state that carries information from 
previous steps — t